# ADS1001 Project Report Notebook

## 1. Title, Members & Theme

**Project title:** Communicating Useful Weather Indicators for Public Weather Guidance in Malaysia

**Group members:**

- Member 1: ____________________
- Member 2: ____________________
- Member 3: ____________________
- Member 4: ____________________
- Member 5: ____________________

**Theme and client context:** This project supports a Malaysian public weather information service. The goal is to identify which weather indicators should be prioritised for public guidance, warnings, and simple day-to-day communication.

**Integrated source notebooks:**

| Source notebook | Contribution to this final notebook |
|---|---|
| `malaysia_weather_data_cleaning_process.ipynb` | Shared loading, data audit, cleaning logic, validation checks |
| `ADS1001 notebook Q1.ipynb` | Temperature versus wind chill analysis |
| `weather_project.ipynb` | Humidity, dew point, and oppressive weather analysis |
| `ADS1001_updated_with_cleaned_data.ipynb` | Atmospheric pressure, pressure trend, and rainfall analysis |
| `xu_zihao_uv_index_analysis.ipynb` | UV Index risk categories and public warning thresholds |


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

try:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        average_precision_score,
        confusion_matrix,
    )
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

PROJECT_ROOT = Path.cwd()


## 2. Introduction & Research Questions

Malaysia's weather communication problem is not just about reporting many variables. A public-facing service needs indicators that are understandable, useful, and connected to realistic actions. The same dataset contains temperature, wind chill, humidity, dew point, atmospheric pressure, rainfall, wind, and UV Index readings, but not all of these variables are equally useful for public guidance.

This final notebook integrates the group analyses into one report. The shared objective is to evaluate which weather indicators should be emphasised for different public communication needs.

**Research questions**

1. How different are temperature and wind chill readings in Malaysia, and does wind chill add practical value beyond temperature?
2. How strongly are humidity and dew point associated, and which variable better describes oppressive, sticky weather?
3. How are atmospheric pressure and pressure changes associated with rainfall, and can pressure trend support short-term rain guidance?
4. Which weather conditions correspond to high UV Index values, and what threshold should trigger public UV warnings?

**Overall hypothesis:** Temperature, dew point, pressure trend, rainfall, and UV Index each contribute different information. However, wind chill is expected to add little value in a tropical climate, while dew point and UV Index are expected to be more actionable for public guidance.


In [ ]:
project_questions = pd.DataFrame({
    'Section': [
        'Temperature and wind chill',
        'Humidity and dew point',
        'Pressure and rainfall',
        'UV Index warnings',
    ],
    'Public communication decision': [
        'Should wind chill be highlighted alongside temperature?',
        'Should humidity or dew point be used to describe oppressive weather?',
        'Can pressure trend help flag possible upcoming rain?',
        'When should UV reminders and warnings be issued?',
    ],
    'Main variables': [
        'temperature, wind_chill, wind_speed',
        'temperature, humidity, dew_point, derived heat_index',
        'pressure, pressure_change, precipitation_rate, rain_event',
        'uv_index, temperature, humidity, rainfall, hour',
    ]
})
project_questions


## 3. Data Dictionary

The dataset contains location, weather measurements, and timestamp fields. The table below documents the columns used in the integrated analysis.


In [ ]:
data_dictionary = pd.DataFrame({
    'Column': [
        'place', 'city', 'state', 'temperature', 'pressure', 'dew_point', 'humidity',
        'wind_speed', 'gust', 'wind_chill', 'uv_index', 'precipitation_rate',
        'precipitation_total', 'year', 'month', 'day', 'hour', 'minutes', 'seconds',
        'month_number', 'datetime'
    ],
    'Type': [
        'Categorical', 'Categorical', 'Categorical', 'Numerical', 'Numerical', 'Numerical',
        'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical',
        'Numerical', 'Integer', 'Categorical', 'Integer', 'Integer', 'Integer', 'Integer',
        'Integer', 'Date/Time'
    ],
    'Meaning': [
        'Specific observation place or station label',
        'City where the observation was recorded',
        'State where the observation was recorded',
        'Air temperature in degrees C',
        'Atmospheric pressure in hPa',
        'Dew point temperature in degrees C',
        'Relative humidity percentage',
        'Wind speed',
        'Wind gust speed',
        'Wind chill reading in degrees C',
        'Ultraviolet radiation index',
        'Current precipitation intensity',
        'Accumulated precipitation total',
        'Observation year',
        'Three-letter month label',
        'Day of month',
        'Hour of day, 0 to 23',
        'Minute of observation',
        'Second of observation',
        'Numeric month created during preprocessing',
        'Combined timestamp created during preprocessing'
    ],
    'Role in project': [
        'Location grouping', 'Location grouping', 'Location grouping',
        'Thermal and UV context', 'Rainfall and pressure analysis',
        'Oppressive weather indicator', 'Moisture and heat context',
        'Wind chill and rainfall context', 'Outdoor condition context',
        'Comparison with temperature', 'UV warning analysis',
        'Rainfall event analysis', 'Rainfall event analysis',
        'Time coverage', 'Time coverage', 'Time coverage', 'Hourly patterns',
        'Timestamp construction', 'Timestamp construction', 'Timestamp construction',
        'Time-series analysis'
    ]
})
data_dictionary


## 4. Executive Summary

The integrated analysis supports a clear public communication strategy:

- **Temperature should remain the main thermal headline.** Wind chill is almost identical to temperature in the available Malaysian observations, so it adds little practical value for routine public guidance.
- **Dew point is a stronger comfort signal than humidity alone.** Humidity and dew point are positively associated, but humidity is strongly affected by temperature. Dew point better represents actual moisture in the air and is more useful for describing muggy conditions.
- **Pressure trend can support rain guidance, but it should not be used alone.** Falling pressure has a modest link with next-observation rain probability, while humidity, temperature, time of day, and recent rain patterns also matter. The predictive model is useful as a supporting warning tool rather than a standalone forecast.
- **UV Index needs direct public warnings.** High UV values are concentrated from late morning to mid-afternoon. A public UV warning should begin at **UV Index >= 6**, with a lighter reminder at **UV Index >= 3** and stronger alerts at **UV Index >= 8**.

Overall, the best public-facing weather dashboard should prioritise **temperature, dew point or heat discomfort, rainfall risk, and UV Index**, while treating wind chill and pressure as supporting context.


## 5. Data Loading

The notebook first looks for the dataset inside the project folder so it can be rerun after submission. A desktop path is included only as a fallback for the local working copy.


In [ ]:
EXPECTED_COLUMNS = {
    'place', 'city', 'state', 'temperature', 'pressure', 'dew_point', 'humidity',
    'wind_speed', 'gust', 'wind_chill', 'uv_index', 'precipitation_rate',
    'precipitation_total', 'year', 'month', 'day', 'hour', 'minutes', 'seconds'
}

candidate_paths = [
    PROJECT_ROOT / 'data' / 'malaysia_weather_data.csv',
    PROJECT_ROOT / 'malaysia_weather_data.csv',
    PROJECT_ROOT / 'data' / 'malaysia_weather_cleaned.csv',
    PROJECT_ROOT / 'malaysia_weather_cleaned.csv',
    Path('/Users/c2/Desktop/malaysia_weather_data.csv'),
]


def read_weather_csv(path: Path) -> pd.DataFrame:
    """Read normal CSV files and CSV files that contain one extra title row."""
    for skiprows in [0, 1]:
        try:
            candidate = pd.read_csv(path, skiprows=skiprows, low_memory=False)
        except Exception:
            continue
        if EXPECTED_COLUMNS.issubset(set(candidate.columns)):
            return candidate
    raise ValueError(f'Found {path}, but it does not contain the expected weather columns.')

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Weather CSV not found. Place malaysia_weather_data.csv in the data folder.')

raw_df = read_weather_csv(DATA_PATH)
print(f'Loaded dataset from: {DATA_PATH}')
print(f'Raw dataset shape: {raw_df.shape}')
display(raw_df.head())


## 6. Data Cleaning

The cleaning process follows the shared cleaning notebook, with one extra clarification: missing weather values are not all treated the same way. Rows where **all** weather measurements are missing are unusable and are removed. Remaining missing values are retained in the analysis layer because, for variables such as UV Index and precipitation, missing does not necessarily mean zero.

For modelling only, a separate imputed copy is created later. This keeps the descriptive analysis faithful to the observed data while still allowing the predictive model to run on complete inputs.


In [ ]:
weather_cols = [
    'temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed',
    'gust', 'wind_chill', 'uv_index', 'precipitation_rate', 'precipitation_total'
]
time_cols = ['year', 'month', 'day', 'hour', 'minutes', 'seconds']
location_cols = ['place', 'city', 'state']

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

raw_df = raw_df.copy()
for col in weather_cols + ['year', 'day', 'hour', 'minutes', 'seconds']:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

all_weather_missing = raw_df[weather_cols].isna().all(axis=1)
analysis_base = raw_df.loc[~all_weather_missing].copy().reset_index(drop=True)
analysis_base['month_number'] = analysis_base['month'].map(month_map)
analysis_base['datetime'] = pd.to_datetime(
    {
        'year': analysis_base['year'],
        'month': analysis_base['month_number'],
        'day': analysis_base['day'],
        'hour': analysis_base['hour'],
        'minute': analysis_base['minutes'],
        'second': analysis_base['seconds'],
    },
    errors='coerce'
)

cleaning_overview = pd.DataFrame({
    'Metric': [
        'Rows in raw data',
        'Rows removed because all weather fields were missing',
        'Rows retained for analysis',
        'Columns after adding month_number and datetime',
        'Duplicate rows after cleaning',
        'Invalid datetime values after cleaning',
    ],
    'Value': [
        len(raw_df),
        int(all_weather_missing.sum()),
        len(analysis_base),
        analysis_base.shape[1],
        int(analysis_base.duplicated().sum()),
        int(analysis_base['datetime'].isna().sum()),
    ]
})

missing_after_filter = (
    analysis_base[weather_cols]
    .isna()
    .sum()
    .rename('missing_count')
    .to_frame()
)
missing_after_filter['missing_percent'] = missing_after_filter['missing_count'] / len(analysis_base) * 100

display(cleaning_overview)
display(missing_after_filter.sort_values('missing_count', ascending=False))


## 7. Preprocessing

The analysis uses two related datasets:

- `analysis_base`: cleaned observations with remaining missing values preserved for honest descriptive analysis.
- `model_ready`: an imputed modelling copy with missing-value flags. This is used only where complete predictors are required.

The imputation uses local medians first, then broader medians as fallback. This avoids replacing missing values with arbitrary constants and keeps location, month, and hour context where possible.


In [ ]:
def hierarchical_median_fill(df: pd.DataFrame, col: str) -> pd.Series:
    filled = df[col].copy()
    grouping_levels = [
        ['place', 'month_number', 'hour'],
        ['city', 'month_number', 'hour'],
        ['state', 'month_number', 'hour'],
        ['month_number', 'hour'],
        ['state'],
    ]
    for keys in grouping_levels:
        medians = df.groupby(keys, observed=False)[col].transform('median')
        filled = filled.fillna(medians)
    return filled.fillna(df[col].median())

model_ready = analysis_base.copy()
for col in weather_cols:
    model_ready[f'{col}_missing_flag'] = model_ready[col].isna().astype(int)
    model_ready[col] = hierarchical_median_fill(model_ready, col)

preprocessing_summary = pd.DataFrame({
    'Dataset layer': ['analysis_base', 'model_ready'],
    'Purpose': [
        'Observed-data EDA and variable-specific analyses',
        'Predictive modelling where complete numeric predictors are required',
    ],
    'Rows': [len(analysis_base), len(model_ready)],
    'Missing values in main weather columns': [
        int(analysis_base[weather_cols].isna().sum().sum()),
        int(model_ready[weather_cols].isna().sum().sum()),
    ]
})

preprocessing_summary


## 8. Feature Engineering

Several features are created to make the analyses more interpretable:

- `date` and `day_name` for calendar context.
- `rain_event` to identify records with observed rainfall.
- `pressure_change_1obs` and `pressure_change_3obs` within each location for pressure trend analysis.
- `heat_index` as a warm-weather discomfort proxy.
- UV risk categories and warning flags based on public UV thresholds.


In [ ]:
def add_heat_index(df: pd.DataFrame) -> pd.Series:
    T = df['temperature']
    RH = df['humidity']
    return (
        -8.784695
        + 1.61139411 * T
        + 2.338549 * RH
        - 0.14611605 * T * RH
        - 0.012308094 * T**2
        - 0.016424828 * RH**2
        + 0.002211732 * T**2 * RH
        + 0.00072546 * T * RH**2
        - 0.000003582 * T**2 * RH**2
    )

for frame in [analysis_base, model_ready]:
    frame['date'] = frame['datetime'].dt.date
    frame['day_name'] = frame['datetime'].dt.day_name()
    frame['heat_index'] = add_heat_index(frame)
    frame['oppressive_dew_point'] = frame['dew_point'] >= 24
    frame['high_humidity'] = frame['humidity'] >= 80
    frame['uv_warning_ge6'] = frame['uv_index'] >= 6
    frame['uv_very_high_ge8'] = frame['uv_index'] >= 8

pressure_sort_cols = ['state', 'city', 'place', 'datetime']
model_ready = model_ready.sort_values(pressure_sort_cols).reset_index(drop=True)
model_ready['rain_event'] = (
    (model_ready['precipitation_rate'] > 0) |
    (model_ready['precipitation_total'] > 0)
).astype(int)
model_ready['pressure_change_1obs'] = model_ready.groupby(location_cols)['pressure'].diff()
model_ready['pressure_change_3obs'] = model_ready.groupby(location_cols)['pressure'].diff(3)
model_ready['next_rain_event'] = model_ready.groupby(location_cols)['rain_event'].shift(-1)
model_ready['falling_pressure'] = (model_ready['pressure_change_1obs'] < 0).astype(int)
model_ready['hour_sin'] = np.sin(2 * np.pi * model_ready['hour'] / 24)
model_ready['hour_cos'] = np.cos(2 * np.pi * model_ready['hour'] / 24)

feature_summary = pd.DataFrame({
    'Feature': [
        'month_number', 'datetime', 'heat_index', 'oppressive_dew_point',
        'high_humidity', 'pressure_change_1obs', 'pressure_change_3obs',
        'next_rain_event', 'uv_warning_ge6', 'uv_very_high_ge8'
    ],
    'Reason': [
        'Converts month labels into sortable numeric values',
        'Supports time coverage and trend analysis',
        'Combines temperature and humidity into a heat discomfort proxy',
        'Flags dew point readings linked with muggy conditions',
        'Compares a common humidity threshold with dew point',
        'Measures short-term pressure direction within location',
        'Measures a broader recent pressure tendency within location',
        'Targets upcoming rainfall for the predictive model',
        'Client-facing high UV warning threshold',
        'Client-facing very high UV alert threshold'
    ]
})
feature_summary


## 9. Preliminary Analysis / Exploratory Data Analysis (EDA)

This section checks the shape, coverage, missingness, and broad relationships before answering the four research questions.


In [ ]:
dataset_profile = pd.DataFrame({
    'Metric': [
        'Cleaned rows', 'Columns', 'States', 'Cities', 'Places',
        'Earliest timestamp', 'Latest timestamp', 'Distinct observation dates'
    ],
    'Value': [
        len(analysis_base),
        analysis_base.shape[1],
        analysis_base['state'].nunique(),
        analysis_base['city'].nunique(),
        analysis_base['place'].nunique(),
        analysis_base['datetime'].min(),
        analysis_base['datetime'].max(),
        analysis_base['datetime'].dt.date.nunique(),
    ]
})

display(dataset_profile)

descriptive_stats = analysis_base[weather_cols].describe().T

display(descriptive_stats)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
missing_after_filter.sort_values('missing_percent')['missing_percent'].plot(kind='barh', ax=axes[0])
axes[0].set_title('Missing values after removing unusable rows')
axes[0].set_xlabel('Missing percentage')
axes[0].set_ylabel('Weather variable')

corr = analysis_base[weather_cols].corr(numeric_only=True)
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', vmin=-1, vmax=1, center=0)
axes[1].set_title('Correlation among weather measurements')

plt.tight_layout()
plt.show()


## 10. Conclusions from EDA

The dataset is suitable for integrated weather-indicator analysis after removing rows with no weather measurements. There are no duplicate rows and the combined timestamps are valid. The remaining missing values are concentrated in variables that are often harder to observe continuously, especially UV Index, gust, and precipitation.

This EDA leads to three important modelling decisions:

1. Use variable-specific row filtering for descriptive analyses so that missing UV or rainfall values are not confused with zero values.
2. Use an imputed copy only for the predictive model, with missing flags retained.
3. Interpret correlations as associations, not causal evidence.


## 11. Modeling / Analysis

### 11.1 Temperature and Wind Chill

This section answers whether wind chill provides additional public communication value beyond temperature in Malaysia.


In [ ]:
wind_df = analysis_base[
    ['place', 'city', 'state', 'datetime', 'temperature', 'wind_chill', 'wind_speed']
].dropna(subset=['temperature', 'wind_chill']).copy()
wind_df['difference'] = wind_df['temperature'] - wind_df['wind_chill']
wind_df['abs_difference'] = wind_df['difference'].abs()

wind_metrics = pd.DataFrame({
    'Metric': [
        'Valid rows used',
        'Mean temperature',
        'Mean wind chill',
        'Mean difference: temperature - wind chill',
        'Median difference',
        'Correlation',
        'Rows with exact equality',
        'Rows with abs difference <= 0.5 degrees C',
        'Rows with abs difference <= 1.0 degrees C',
        'Rows with abs difference > 2.0 degrees C',
        'Maximum absolute difference',
        'Rows with non-zero difference',
    ],
    'Value': [
        len(wind_df),
        wind_df['temperature'].mean(),
        wind_df['wind_chill'].mean(),
        wind_df['difference'].mean(),
        wind_df['difference'].median(),
        wind_df['temperature'].corr(wind_df['wind_chill']),
        (wind_df['abs_difference'] == 0).mean(),
        (wind_df['abs_difference'] <= 0.5).mean(),
        (wind_df['abs_difference'] <= 1.0).mean(),
        (wind_df['abs_difference'] > 2.0).mean(),
        wind_df['abs_difference'].max(),
        int((wind_df['abs_difference'] > 0).sum()),
    ]
})

wind_metrics_display = wind_metrics.copy()
percent_rows = [6, 7, 8, 9]
wind_metrics_display['Value'] = wind_metrics_display['Value'].astype(object)
for idx in percent_rows:
    wind_metrics_display.loc[idx, 'Value'] = f"{wind_metrics.loc[idx, 'Value'] * 100:.4f}%"
for idx in [1, 2, 3, 4, 5, 10]:
    wind_metrics_display.loc[idx, 'Value'] = f"{wind_metrics.loc[idx, 'Value']:.4f}"

display(wind_metrics_display)

wind_df['wind_bin'] = pd.cut(
    wind_df['wind_speed'],
    bins=[-0.01, 1, 3, 5, 8, np.inf],
    labels=['0-1', '1-3', '3-5', '5-8', '8+']
)
wind_summary = (
    wind_df.dropna(subset=['wind_bin'])
    .groupby('wind_bin', observed=False)['abs_difference']
    .agg(observations='size', mean_abs_difference='mean', max_abs_difference='max')
    .reset_index()
)
display(wind_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
axes[0].hist(wind_df['temperature'], bins=30, alpha=0.6, label='Temperature')
axes[0].hist(wind_df['wind_chill'], bins=30, alpha=0.6, label='Wind chill')
axes[0].set_title('Temperature and wind chill distributions')
axes[0].set_xlabel('Degrees C')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].scatter(wind_df['temperature'], wind_df['wind_chill'], s=8, alpha=0.25)
min_val = min(wind_df['temperature'].min(), wind_df['wind_chill'].min())
max_val = max(wind_df['temperature'].max(), wind_df['wind_chill'].max())
axes[1].plot([min_val, max_val], [min_val, max_val], linestyle='--', color='black')
axes[1].set_title('Temperature vs wind chill')
axes[1].set_xlabel('Temperature')
axes[1].set_ylabel('Wind chill')

axes[2].bar(wind_summary['wind_bin'].astype(str), wind_summary['mean_abs_difference'])
axes[2].set_title('Mean absolute difference by wind speed')
axes[2].set_xlabel('Wind speed band')
axes[2].set_ylabel('Mean absolute difference')

plt.tight_layout()
plt.show()

non_zero_wind = wind_df.loc[wind_df['abs_difference'] > 0].sort_values('abs_difference', ascending=False)
display(non_zero_wind.head(10))


**Temperature and wind chill conclusion:** The two variables are practically identical in the observed data. Wind chill has near-perfect correlation with temperature, and almost every valid record has exactly the same value for both variables. For a Malaysian public weather service, wind chill should not be a headline thermal indicator. Temperature should be shown instead, while warm-weather comfort should be explained using humidity, dew point, or heat-index-style measures.


### 11.2 Humidity, Dew Point, and Oppressive Weather

This section compares humidity and dew point as indicators of muggy or oppressive weather.


In [ ]:
humidity_df = analysis_base[
    ['datetime', 'city', 'state', 'temperature', 'humidity', 'dew_point']
].dropna(subset=['temperature', 'humidity', 'dew_point']).copy()
humidity_df['heat_index'] = add_heat_index(humidity_df)
humidity_df['oppressive_dew_point'] = humidity_df['dew_point'] >= 24
humidity_df['high_humidity'] = humidity_df['humidity'] >= 80

humidity_summary = humidity_df[['temperature', 'humidity', 'dew_point']].describe().T

dew_humidity_metrics = pd.DataFrame({
    'Metric': [
        'Valid rows used',
        'Mean humidity',
        'Mean dew point',
        'Pearson correlation: humidity vs dew point',
        'Spearman correlation: humidity vs dew point',
        'Share with humidity >= 80%',
        'Share with dew point >= 24 degrees C',
        'Share meeting both thresholds',
    ],
    'Value': [
        len(humidity_df),
        humidity_df['humidity'].mean(),
        humidity_df['dew_point'].mean(),
        humidity_df['humidity'].corr(humidity_df['dew_point']),
        humidity_df['humidity'].corr(humidity_df['dew_point'], method='spearman'),
        humidity_df['high_humidity'].mean(),
        humidity_df['oppressive_dew_point'].mean(),
        (humidity_df['high_humidity'] & humidity_df['oppressive_dew_point']).mean(),
    ]
})

dew_humidity_display = dew_humidity_metrics.copy()
dew_humidity_display['Value'] = dew_humidity_display['Value'].astype(object)
for idx in [5, 6, 7]:
    dew_humidity_display.loc[idx, 'Value'] = f"{dew_humidity_metrics.loc[idx, 'Value'] * 100:.2f}%"
for idx in [1, 2, 3, 4]:
    dew_humidity_display.loc[idx, 'Value'] = f"{dew_humidity_metrics.loc[idx, 'Value']:.3f}"

display(humidity_summary)
display(dew_humidity_display)

warm_weather = humidity_df[(humidity_df['temperature'] >= 27) & (humidity_df['humidity'] >= 40)].copy()
heat_index_corr = warm_weather[['heat_index', 'humidity', 'dew_point', 'temperature']].corr()
display(heat_index_corr)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(humidity_df['humidity'], bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of humidity')
axes[0, 0].set_xlabel('Humidity (%)')

sns.histplot(humidity_df['dew_point'], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of dew point')
axes[0, 1].set_xlabel('Dew point (degrees C)')

sns.scatterplot(data=humidity_df, x='dew_point', y='humidity', alpha=0.25, s=14, ax=axes[1, 0])
axes[1, 0].set_title('Humidity vs dew point')
axes[1, 0].set_xlabel('Dew point (degrees C)')
axes[1, 0].set_ylabel('Humidity (%)')

sns.heatmap(heat_index_corr, annot=True, vmin=-1, vmax=1, cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Correlation with heat index')

plt.tight_layout()
plt.show()

state_comfort_summary = (
    humidity_df.groupby('state')
    .agg(
        records=('humidity', 'size'),
        mean_temperature=('temperature', 'mean'),
        mean_humidity=('humidity', 'mean'),
        mean_dew_point=('dew_point', 'mean'),
        oppressive_dew_point_rate=('oppressive_dew_point', 'mean'),
    )
    .sort_values('mean_dew_point', ascending=False)
)
display(state_comfort_summary)


**Humidity and dew point conclusion:** Humidity and dew point are related, but not interchangeable. Relative humidity is strongly shaped by temperature, while dew point more directly reflects moisture in the air. A dew point threshold around 24 degrees C identifies many observations as muggy or oppressive, and dew point has a stronger positive relationship with heat index than humidity alone. For public messaging, dew point is the better single moisture indicator, while humidity remains useful supporting context.


### 11.3 Pressure, Pressure Trend, and Rainfall

This section improves the pressure analysis by calculating pressure change within each location and by separating current rainfall from **next-observation rainfall**. This better matches the question of whether falling pressure can support early rainfall guidance.


In [ ]:
pressure_df = analysis_base.dropna(
    subset=['pressure', 'precipitation_rate', 'temperature', 'humidity', 'dew_point', 'wind_speed']
).copy()
pressure_df = pressure_df.sort_values(pressure_sort_cols).reset_index(drop=True)
pressure_df['rain_event'] = (
    (pressure_df['precipitation_rate'] > 0) |
    (pressure_df['precipitation_total'].fillna(0) > 0)
).astype(int)
pressure_df['pressure_change_1obs'] = pressure_df.groupby(location_cols)['pressure'].diff()
pressure_df['pressure_change_3obs'] = pressure_df.groupby(location_cols)['pressure'].diff(3)
pressure_df['next_rain_event'] = pressure_df.groupby(location_cols)['rain_event'].shift(-1)
pressure_df['pressure_trend'] = np.where(pressure_df['pressure_change_1obs'] < 0, 'Falling', 'Stable/Rising')

pressure_analysis_df = pressure_df.dropna(subset=['pressure_change_1obs', 'next_rain_event']).copy()
pressure_analysis_df['next_rain_event'] = pressure_analysis_df['next_rain_event'].astype(int)

pressure_stats = pressure_analysis_df[
    ['pressure', 'pressure_change_1obs', 'pressure_change_3obs', 'precipitation_rate']
].describe().T

trend_summary = (
    pressure_analysis_df.groupby('pressure_trend')
    .agg(
        records=('rain_event', 'size'),
        current_rain_rate=('rain_event', 'mean'),
        next_observation_rain_rate=('next_rain_event', 'mean'),
        mean_pressure=('pressure', 'mean'),
        mean_pressure_change=('pressure_change_1obs', 'mean'),
    )
)
trend_display = trend_summary.copy()
for col in ['current_rain_rate', 'next_observation_rain_rate']:
    trend_display[col] = trend_display[col].map(lambda x: f'{x:.2%}')

display(pressure_stats)
display(trend_display)

pressure_corr = pressure_analysis_df[[
    'pressure', 'pressure_change_1obs', 'pressure_change_3obs',
    'temperature', 'humidity', 'wind_speed', 'precipitation_rate',
    'rain_event', 'next_rain_event'
]].corr(numeric_only=True)
display(pressure_corr[['precipitation_rate', 'rain_event', 'next_rain_event']])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=pressure_analysis_df, x='rain_event', y='pressure', ax=axes[0])
axes[0].set_title('Pressure distribution by current rain')
axes[0].set_xlabel('Current rain event')
axes[0].set_ylabel('Pressure')

sns.boxplot(data=pressure_analysis_df, x='next_rain_event', y='pressure_change_1obs', ax=axes[1])
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Pressure change by next rain event')
axes[1].set_xlabel('Next-observation rain event')
axes[1].set_ylabel('Pressure change from previous observation')

trend_summary['next_observation_rain_rate'].plot(kind='bar', ax=axes[2], color=['#4c78a8', '#e45756'])
axes[2].set_title('Next-observation rain probability by pressure trend')
axes[2].set_ylabel('Rain probability')
axes[2].set_ylim(0, 1)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
model_results = None
feature_importance = None

if SKLEARN_AVAILABLE:
    model_features = [
        'pressure', 'pressure_change_1obs', 'pressure_change_3obs',
        'temperature', 'humidity', 'dew_point', 'wind_speed',
        'falling_pressure', 'hour_sin', 'hour_cos'
    ]
    rain_model_df = model_ready.dropna(subset=model_features + ['next_rain_event']).copy()
    rain_model_df['next_rain_event'] = rain_model_df['next_rain_event'].astype(int)

    X = rain_model_df[model_features]
    y = rain_model_df['next_rain_event']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    rain_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight='balanced')
    )
    rain_model.fit(X_train, y_train)

    y_probability = rain_model.predict_proba(X_test)[:, 1]
    y_pred = (y_probability >= 0.50).astype(int)
    baseline_pred = np.zeros_like(y_test)

    model_results = pd.DataFrame({
        'Metric': [
            'Test rows', 'Positive class rate in test set',
            'Model accuracy', 'Model precision', 'Model recall', 'Model F1',
            'Model ROC-AUC', 'Model average precision',
            'No-rain baseline accuracy', 'No-rain baseline recall'
        ],
        'Value': [
            len(y_test),
            y_test.mean(),
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred, zero_division=0),
            recall_score(y_test, y_pred, zero_division=0),
            f1_score(y_test, y_pred, zero_division=0),
            roc_auc_score(y_test, y_probability),
            average_precision_score(y_test, y_probability),
            accuracy_score(y_test, baseline_pred),
            recall_score(y_test, baseline_pred, zero_division=0),
        ]
    })

    display(model_results)

    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred),
        index=['Actual no next rain', 'Actual next rain'],
        columns=['Predicted no next rain', 'Predicted next rain']
    )
    display(cm)

    logistic_model = rain_model.named_steps['logisticregression']
    feature_importance = (
        pd.Series(logistic_model.coef_[0], index=model_features, name='standardised_logistic_coefficient')
        .sort_values(key=lambda s: s.abs(), ascending=False)
        .to_frame()
    )
    display(feature_importance)
else:
    print('scikit-learn is not installed, so the rainfall model was skipped.')


**Pressure and rainfall conclusion:** Pressure has a relationship with rainfall, but the pattern is modest rather than strong. Falling pressure is more useful for thinking about the next observation than the current observation, which fits the idea of pressure trend as an early signal. However, the predictive model shows that pressure trend should be combined with humidity, temperature, wind, and time features. It should be communicated as a supporting rainfall-risk indicator, not as a standalone forecast.


### 11.4 UV Index Conditions and Warning Thresholds

This section keeps missing UV values as missing. A missing UV reading is not the same as UV Index 0. Only valid UV readings are used for UV risk categories and warning thresholds.


In [ ]:
uv_df = analysis_base.dropna(subset=['uv_index']).copy()
uv_bins = [-0.1, 2, 5, 7, 10, np.inf]
uv_labels = ['Low (0-2)', 'Moderate (3-5)', 'High (6-7)', 'Very high (8-10)', 'Extreme (11+)']
uv_df['uv_category'] = pd.cut(uv_df['uv_index'], bins=uv_bins, labels=uv_labels)
uv_df['daylight_uv'] = uv_df['uv_index'] > 0
uv_df['sun_protection_needed'] = uv_df['uv_index'] >= 3
uv_df['public_warning'] = uv_df['uv_index'] >= 6
uv_df['very_high_alert'] = uv_df['uv_index'] >= 8
uv_df['extreme_alert'] = uv_df['uv_index'] >= 11
uv_df['wet_weather'] = uv_df['precipitation_rate'].fillna(0) > 0

uv_cleaning_summary = pd.DataFrame({
    'Metric': [
        'Rows with valid UV Index',
        'Rows with missing UV Index',
        'Rows where UV Index = 0',
        'Rows where UV Index > 0',
        'Rows where UV Index >= 6',
        'Rows where UV Index >= 8',
        'Maximum UV Index',
    ],
    'Value': [
        len(uv_df),
        int(analysis_base['uv_index'].isna().sum()),
        int((uv_df['uv_index'] == 0).sum()),
        int(uv_df['daylight_uv'].sum()),
        int(uv_df['public_warning'].sum()),
        int(uv_df['very_high_alert'].sum()),
        uv_df['uv_index'].max(),
    ]
})

display(uv_cleaning_summary)

category_summary = (
    uv_df.groupby('uv_category', observed=False)
    .agg(
        records=('uv_index', 'size'),
        average_uv=('uv_index', 'mean'),
        median_uv=('uv_index', 'median')
    )
    .reset_index()
)
daylight_counts = (
    uv_df.loc[uv_df['daylight_uv']]
    .groupby('uv_category', observed=False)
    .size()
    .reindex(category_summary['uv_category'])
    .fillna(0)
    .astype(int)
    .to_numpy()
)
category_summary['daylight_records'] = daylight_counts
category_summary['share_of_valid_uv_records'] = category_summary['records'] / len(uv_df)
category_summary['share_of_daylight_uv_records'] = category_summary['daylight_records'] / uv_df['daylight_uv'].sum()

category_display = category_summary.copy()
for col in ['share_of_valid_uv_records', 'share_of_daylight_uv_records']:
    category_display[col] = category_display[col].map(lambda x: f'{x:.1%}')
display(category_display)

hour_summary = (
    uv_df.groupby('hour')
    .agg(
        records=('uv_index', 'size'),
        mean_uv=('uv_index', 'mean'),
        median_uv=('uv_index', 'median'),
        max_uv=('uv_index', 'max'),
        warning_rate_ge6=('public_warning', 'mean'),
        very_high_rate_ge8=('very_high_alert', 'mean')
    )
    .reset_index()
)
main_warning_window = hour_summary[hour_summary['hour'].between(10, 16)].copy()
display(main_warning_window)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(category_summary['uv_category'].astype(str), category_summary['records'])
axes[0].set_title('UV records by risk category')
axes[0].set_xlabel('UV category')
axes[0].set_ylabel('Records')
axes[0].tick_params(axis='x', rotation=25)

axes[1].plot(hour_summary['hour'], hour_summary['mean_uv'], marker='o')
axes[1].axhline(3, linestyle='--', color='orange', label='Reminder: UV >= 3')
axes[1].axhline(6, linestyle='--', color='red', label='Warning: UV >= 6')
axes[1].set_title('Average UV Index by hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Average UV Index')
axes[1].legend()

axes[2].plot(hour_summary['hour'], hour_summary['warning_rate_ge6'], marker='o', color='red')
axes[2].set_title('Warning rate by hour')
axes[2].set_xlabel('Hour')
axes[2].set_ylabel('Share with UV >= 6')
axes[2].set_ylim(0, 1)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()


In [ ]:
condition_summary = (
    uv_df.groupby('uv_category', observed=False)
    .agg(
        records=('uv_index', 'size'),
        median_uv=('uv_index', 'median'),
        mean_temperature=('temperature', 'mean'),
        median_temperature=('temperature', 'median'),
        mean_humidity=('humidity', 'mean'),
        median_humidity=('humidity', 'median'),
        mean_dew_point=('dew_point', 'mean'),
        median_pressure=('pressure', 'median'),
        median_wind_speed=('wind_speed', 'median'),
        wet_weather_rate=('wet_weather', 'mean'),
        median_hour=('hour', 'median'),
    )
    .reset_index()
)
condition_display = condition_summary.copy()
condition_display['wet_weather_rate'] = condition_display['wet_weather_rate'].map(lambda x: f'{x:.1%}')
display(condition_display)

corr_vars = [
    'uv_index', 'temperature', 'humidity', 'dew_point', 'pressure', 'wind_speed',
    'gust', 'precipitation_rate', 'precipitation_total', 'hour', 'month_number'
]
correlation_with_uv = (
    uv_df[corr_vars]
    .corr(numeric_only=True)['uv_index']
    .drop('uv_index')
    .sort_values()
)
display(correlation_with_uv.to_frame('correlation_with_uv_index'))

uv_df['temperature_bin'] = pd.cut(
    uv_df['temperature'],
    bins=[-np.inf, 25, 28, 31, 34, np.inf],
    labels=['<=25', '25-28', '28-31', '31-34', '>34']
)
uv_df['humidity_bin'] = pd.cut(
    uv_df['humidity'],
    bins=[-np.inf, 60, 70, 80, 90, np.inf],
    labels=['<=60', '60-70', '70-80', '80-90', '>90']
)
risk_grid = (
    uv_df.dropna(subset=['temperature_bin', 'humidity_bin'])
    .groupby(['humidity_bin', 'temperature_bin'], observed=False)['public_warning']
    .mean()
    .unstack()
)

threshold_rows = []
for threshold, label, client_action in [
    (3, 'Moderate or above', 'Show sun-protection reminder'),
    (6, 'High or above', 'Issue public UV warning'),
    (8, 'Very high or above', 'Escalate warning / extra protection'),
    (11, 'Extreme', 'Urgent extreme UV alert'),
]:
    mask = uv_df['uv_index'] >= threshold
    threshold_rows.append({
        'threshold': f'UV Index >= {threshold}',
        'category_level': label,
        'records': int(mask.sum()),
        'share_of_valid_uv_records': mask.mean(),
        'share_of_daylight_uv_records': mask.sum() / uv_df['daylight_uv'].sum(),
        'share_occurring_between_10_and_16': uv_df.loc[mask, 'hour'].between(10, 16).mean(),
        'recommended_client_action': client_action,
    })
threshold_summary = pd.DataFrame(threshold_rows)
threshold_display = threshold_summary.copy()
for col in ['share_of_valid_uv_records', 'share_of_daylight_uv_records', 'share_occurring_between_10_and_16']:
    threshold_display[col] = threshold_display[col].map(lambda x: f'{x:.1%}')
display(threshold_display)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
correlation_with_uv.plot(kind='barh', ax=axes[0])
axes[0].set_title('Correlation with UV Index')
axes[0].set_xlabel('Correlation')
axes[0].set_ylabel('Variable')

image = axes[1].imshow(risk_grid, aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Probability of UV warning by temperature and humidity')
axes[1].set_xlabel('Temperature band')
axes[1].set_ylabel('Humidity band')
axes[1].set_xticks(range(len(risk_grid.columns)))
axes[1].set_yticks(range(len(risk_grid.index)))
axes[1].set_xticklabels(risk_grid.columns)
axes[1].set_yticklabels(risk_grid.index)
for row in range(risk_grid.shape[0]):
    for col in range(risk_grid.shape[1]):
        value = risk_grid.iloc[row, col]
        if pd.notna(value):
            axes[1].text(col, row, f'{value:.0%}', ha='center', va='center')
fig.colorbar(image, ax=axes[1], label='Warning probability')

plt.tight_layout()
plt.show()


**UV Index conclusion:** The highest UV values are mainly associated with late morning to mid-afternoon conditions, warmer temperatures, lower relative humidity than low-UV periods, and little current rain. These variables describe the context around high UV, but they do not replace direct UV Index reporting. The recommended public warning threshold is **UV Index >= 6**, with a general reminder at **UV Index >= 3** and stronger alerts at **UV Index >= 8**.


## 12. Model / Analysis Conclusions

The table below turns the analysis into client-facing recommendations.


In [ ]:
pressure_next_falling = trend_summary.loc['Falling', 'next_observation_rain_rate'] if 'Falling' in trend_summary.index else np.nan
pressure_next_stable = trend_summary.loc['Stable/Rising', 'next_observation_rain_rate'] if 'Stable/Rising' in trend_summary.index else np.nan
rain_auc = None
if model_results is not None:
    rain_auc = float(model_results.loc[model_results['Metric'] == 'Model ROC-AUC', 'Value'].iloc[0])

recommendations = pd.DataFrame({
    'Indicator': ['Temperature', 'Wind chill', 'Dew point', 'Humidity', 'Pressure trend', 'UV Index'],
    'Evidence from analysis': [
        f"Mean temperature is {wind_df['temperature'].mean():.2f} degrees C in the wind-chill comparison sample.",
        f"{(wind_df['abs_difference'] == 0).mean() * 100:.2f}% of valid temperature/wind-chill records are exactly equal.",
        f"{humidity_df['oppressive_dew_point'].mean() * 100:.2f}% of valid records have dew point >= 24 degrees C.",
        f"Humidity-dew point Pearson correlation is {humidity_df['humidity'].corr(humidity_df['dew_point']):.3f}, so they are related but not identical.",
        f"Next-observation rain probability is {pressure_next_falling:.2%} after falling pressure versus {pressure_next_stable:.2%} after stable/rising pressure; model ROC-AUC is {rain_auc:.3f}.",
        f"UV >= 6 occurs in {threshold_summary.loc[threshold_summary['threshold'] == 'UV Index >= 6', 'share_of_valid_uv_records'].iloc[0]:.1%} of valid UV records and is concentrated between 10:00 and 16:00.",
    ],
    'Recommendation': [
        'Use as the main thermal headline.',
        'Do not emphasise for routine Malaysian public guidance.',
        'Use as the main moisture/discomfort indicator.',
        'Keep as supporting context because it is temperature-dependent.',
        'Use as a supporting rainfall-risk signal, not a standalone forecast.',
        'Always show in daylight when available and issue warnings at UV >= 6.',
    ]
})
recommendations


## 13. Overall Conclusions

The group project shows that a useful Malaysian public weather service should not treat every weather variable as equally important. The strongest public-facing indicators are those that either communicate immediate conditions clearly or lead to a practical action.

**Answers to the research questions:**

1. **Temperature and wind chill:** Wind chill adds little value because it is almost always equal to temperature in this tropical dataset. Temperature should remain the main thermal indicator.
2. **Humidity and dew point:** Dew point is more useful than humidity alone for describing oppressive weather because it reflects actual moisture in the air and relates more clearly to heat discomfort.
3. **Pressure and rainfall:** Pressure trend has some value for upcoming rain guidance, especially when combined with other weather variables. It should be used as supporting evidence, not as a single warning trigger.
4. **UV Index:** High UV is concentrated around late morning and early afternoon. Public communication should use UV Index directly, with reminders at UV >= 3, warnings at UV >= 6, and stronger alerts at UV >= 8.

**Final client recommendation:** A public weather dashboard for Malaysia should prioritise temperature, dew point or heat discomfort, rainfall risk, and UV Index. Wind chill can be de-emphasised, while pressure trend can appear as a supporting rainfall context variable.


## 14. Limitations

- The dataset covers the available observations only and does not represent every Malaysian location equally.
- Some important meteorological variables, such as cloud cover, solar radiation, monsoon phase, and direct human comfort ratings, are not available.
- The pressure model predicts next-observation rain, but observation gaps are not always equal, so it is a practical indicator rather than a precise forecast model.
- Correlation and logistic regression show associations, not causal relationships.
- Missing values are handled differently depending on the question. This is intentional, but it means row counts vary across sections.
- UV warning thresholds are based on standard public health categories, but a real public warning system should verify timezone handling, exposure duration, and local operational requirements before deployment.


## 15. References

- World Health Organization (2022) *Radiation: The ultraviolet (UV) index*. Available at: https://www.who.int/news-room/questions-and-answers/item/radiation-the-ultraviolet-%28uv%29-index
- World Health Organization (2022) *Ultraviolet radiation*. Available at: https://www.who.int/news-room/fact-sheets/detail/ultraviolet-radiation
- United States Environmental Protection Agency (2026) *UV Index Scale*. Available at: https://www.epa.gov/sunsafety/uv-index-scale-0
- National Environment Agency Singapore (n.d.) *Ultraviolet Index*. Available at: https://www.nea.gov.sg/corporate-functions/weather/ultraviolet-index
